In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df = {}

df["train_orig"] = pd.read_csv("train.csv")
df["test_orig"]  = pd.read_csv("test.csv")

df["total_orig"] = pd.concat(objs=[df["train_orig"], df["test_orig"]], ignore_index=True)

In [3]:
df["total_orig"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
dtypes: float64(3), int64(4), object(5)
memory usage: 122.8+ KB


In [4]:
def create_total():
    df = globals()["df"]
    incomplete_columns = ["Cabin"]
    df["total"] = df["total_orig"].drop(labels=incomplete_columns, axis=1)
    
    unhelpful_columns = ["PassengerId", "Name", "Ticket"]

    df["total"] = df["total"].drop(labels=unhelpful_columns, axis=1)
    
    # drop_na_columns = ["Embarked"]
    drop_na_columns = []

    for column in drop_na_columns:
        df["total"] = df["total"][~df["total"][column].isna()]
        
    df["total"] = pd.get_dummies(df["total"], columns=["Sex", "Embarked"])

In [5]:
from sklearn.model_selection import train_test_split

def run_clf(clf, column = "Survived", drop_na_columns=False):
    df_total = globals()["df"]["total"]
    columns_with_na = df_total.columns[df_total.isna().any()]
    train    = df_total[~df_total[column].isna()]
    test     = df_total[ df_total[column].isna()].drop(column, axis=1)

    if drop_na_columns:        
        columns_with_na = test.columns[test.isna().any()]
        train = train.drop(columns_with_na, axis=1)
        test  = test.drop(columns_with_na, axis=1)
    else:
        train = train.dropna()
        test  = test.dropna()

    X_train = train.drop(column, axis=1)
    y_train = train[column]
    X_test  = test.copy()
    
    if X_test.shape[0] == 0: return []

    computed_column_name = column + "_computed"
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    df_total[computed_column_name] = df_total[column]
#     import IPython; IPython.embed(using=None)
    df_total.loc[X_test.index, computed_column_name] = y_pred
    df_total.drop(column, axis=1, inplace=True)

    X_train2, X_test2, y_train2, y_test2 = train_test_split(X_train, y_train, test_size=0.33, random_state=42)
    
    acc_log = round(clf.score(X_train2, y_train2) * 100, 2)
    print(acc_log)
    return y_pred

In [6]:
import xgboost as xgb

def impute_missing():
    run_clf(xgb.XGBRegressor(), "Age")
    run_clf(xgb.XGBRegressor(), "Age_computed", drop_na_columns=True)
    
    run_clf(xgb.XGBRegressor(), "Fare")
    run_clf(xgb.XGBRegressor(), "Fare", drop_na_columns=True)

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB


random_state = 141

estimators = {
    "Random Forest":       RandomForestClassifier(n_estimators=1000, max_depth=5, random_state=random_state),
    "K Nearest Neighbors": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(random_state=random_state),
    "Linear SVC":          LinearSVC(random_state=random_state),
    "Decision Tree":       DecisionTreeClassifier(random_state=random_state, max_depth=5),
    "Naive-Bayes":         GaussianNB(),
    "XGBoost Classifier":  xgb.XGBClassifier(max_depth=5, random_state=random_state),
    
}

for name, classifier in estimators.items():
    create_total()
    impute_missing()
    result = run_clf(classifier, "Survived")
    print(f"^^^^ {name} ^^^^\n")

# run_clf(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1), "Survived")
# run_clf(KNeighborsClassifier(), "Survived")

76.26
70.57
88.98
85.57
^^^^ Random Forest ^^^^

76.26
70.57
88.98
80.54
^^^^ K Nearest Neighbors ^^^^

76.26
70.57
88.98
79.7
^^^^ Logistic Regression ^^^^



/Users/peter.maneykowski/.pyenv/versions/3.9.13/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:444: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


76.26
70.57
88.98
70.47
^^^^ Linear SVC ^^^^



/Users/peter.maneykowski/.pyenv/versions/3.9.13/lib/python3.9/site-packages/sklearn/svm/_base.py:1225: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


76.26
70.57
88.98
84.73
^^^^ Decision Tree ^^^^

76.26
70.57
88.98
78.86
^^^^ Naive-Bayes ^^^^

76.26
70.57
88.98
96.81
^^^^ XGBoost Classifier ^^^^



In [8]:
# clf = RandomForestClassifier(n_estimators=1000, random_state=random_state)

clf = xgb.XGBClassifier(max_depth=5, random_state=random_state)

create_total()
impute_missing()
run_clf(clf, "Survived")

76.26
70.57
88.98
96.81


array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [9]:
df["total"]["Survived_computed"] = df["total"]["Survived_computed"].round().astype(int)
df["total"]["PassengerId"] = df["total_orig"]["PassengerId"]
test_orig = df["total_orig"][df["total_orig"]["Survived"].isna()]
df["total"].loc[test_orig.index][["PassengerId", "Survived_computed"]].to_csv(
    "submission.csv",
    header=["PassengerId", "Survived"],
    index=False
)